Copyright 2026 Karl Kegel

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

# Compare benchmark runs

Reads the `benchmark.json` of the last runs of one configuration and shows the score of every benchmark and parameter set: the mean over the runs, and the smallest and largest score of a run. The scores are milliseconds per tick, and for `GrowthBenchmark` milliseconds per run.

`runsDir` points at the results that `test_docker.sh --collect "reports/benchmarks"` brings back from the container; a run on the host writes to `execution-engine/build/reports/benchmarks` instead.

In [ ]:
%use serialization, dataframe, kandy

In [ ]:
@Serializable
data class Benchmark(
    val benchmark: String,
    val mode: String,
    val primaryMetric: PrimaryMetric,
    val params: Map<String, String> = emptyMap()
)

@Serializable
data class PrimaryMetric(
    val score: Double,
    val scoreError: Double,
    val scoreUnit: String
)

In [ ]:
// The configuration to read (main, growth, long or smoke) and how many of its latest runs
val configuration = "main"
val runCount = 3
val runsDir = notebook.workingDir.resolve("../../../../build/container/execution-engine/reports/benchmarks/$configuration")

In [ ]:
import kotlin.io.path.isDirectory
import kotlin.io.path.listDirectoryEntries
import kotlin.io.path.name
import kotlin.io.path.readText

// The folders of the runs are named by their time, so the latest sort last
val runFiles = runsDir.listDirectoryEntries()
    .filter { it.isDirectory() }
    .sortedBy { it.name }
    .takeLast(runCount)
    .map { it.resolve("benchmark.json") }

val json = Json { ignoreUnknownKeys = true }
val runs: List<List<Benchmark>> = runFiles.map { json.decodeFromString<List<Benchmark>>(it.readText()) }
runFiles

In [ ]:
fun label(b: Benchmark): String {
    val name = b.benchmark.removePrefix("io.karpfen.benchmark.").substringBefore('.')
    val params = b.params.entries.joinToString(", ") { "${it.key}=${it.value}" }
    return if (params.isEmpty()) name else "$name ($params)"
}

val scores = runs.flatten().groupBy { label(it) }.map { (name, results) ->
    val values = results.map { it.primaryMetric.score }
    Triple(name, values, results.first().primaryMetric.scoreUnit)
}

val table = scores.toDataFrame {
    "benchmark" from { it.first }
    "runs" from { it.second.size }
    "mean" from { it.second.average() }
    "min" from { it.second.min() }
    "max" from { it.second.max() }
    "spreadPercent" from { (it.second.max() - it.second.min()) / it.second.average() * 100 }
    "unit" from { it.third }
}
table

In [ ]:
// One plot per benchmark class, since the scores of the classes lie orders of magnitude apart
for (group in table.groupBy { expr { "benchmark"<String>().substringBefore(' ') } }.groups) {
    val plot = group.plot {
        bars {
            x("benchmark")
            y("mean")
        }
        errorBars {
            x("benchmark")
            yMin("min")
            yMax("max")
            width = 0.2
        }
        coordinatesTransformation = CoordinatesTransformation.cartesianFlipped()
        layout {
            xAxisLabel = ""
            yAxisLabel = "ms"
            size = 700 to (40 * group.rowsCount() + 100)
        }
    }
    DISPLAY(plot)
}